# CBIS 5530 — Build a RAG Chatbot
### MS in AI Strategy (AIST) · Georgia College & State University

---

## Read this before you press anything

This page is a **notebook**. It is a column of boxes, top to bottom. Some boxes are
writing (like this one). Some boxes are code, and those have a **▶ play button** on
the left when you hover over them.

**You do not need to know how to code to use this.** Your job in this course is to
*run* the boxes, *read* what comes out, and *change a few numbers*. That's it.

Three rules:

1. **Go in order, top to bottom.** Later boxes depend on earlier ones. If you skip
   one, the next one will complain.
2. **Wait for the box to finish.** A spinning circle means it's still working. A
   small number in brackets, like `[4]`, means it finished.
3. **Errors are normal and are not your fault.** Red text is information, not
   judgment. Scroll to the *last* line of a red block — that's the actual message.

### If Colab offers to fix an error for you

Colab has a built-in AI assistant that will offer to repair a box that fails. It is
often right, and you are welcome to read what it suggests — but **do not let it paste
settings into a box that didn't have them before.**

The common case: Step 4 fails because you skipped Step 2A, and the assistant fixes it
by copying the chunk settings into Step 4. The error goes away, and the notebook now
has those settings written in two places. Step 11 asks you to change them in Step 2A —
and nothing will happen, because Step 4 is quietly overriding you with its own copy.
That is a genuinely hard bug to see.

If a box gets edited and you're not sure what changed, the safe reset is **File → Save
a copy in Drive** from the original link, and start again.

### Make your own copy first

Go to **File → Save a copy in Drive**. You are now working on your own private
copy and cannot break the original. Do this before anything else.

---

## What you are about to watch happen

By the end of this page you will have built a small retrieval-augmented generation
system and watched it work at every stage. Four things will appear on your screen.
The textbook has a name for each one, and you will be asked for those names:

| You will see | The book calls it | Chapter |
|---|---|---|
| Your document cut into numbered pieces | **chunks** / chunking | 3.2 |
| One piece turned into a long list of numbers | an **embedding** | 2.1 |
| The few pieces that best match your question | **retrieval** | 3.4 |
| The question and those pieces glued together | **context augmentation** | 3.5 |

Those four artifacts are the course. Everything else is plumbing.

---
## Step 1 — Install the tools

Press ▶ on the box below. It takes about 30 seconds and prints a lot of text you
can ignore. You only need to do this once per session.

> If you close this tab and come back tomorrow, you will need to run it again.
> That is normal — the notebook forgets, your work in Drive does not.

In [ ]:
%pip install --quiet openai google-genai pypdf python-docx numpy
print('Done. You can ignore anything above this line.')

---
# Step 2 — Choose a provider and get a key

Your notebook needs permission to use a language model. That permission comes from a
**provider**, and it takes the form of a long password called an **API key**.

First, one thing that confuses almost everyone:

> **A ChatGPT subscription is not an API key.** They are separate systems with separate
> billing. Paying for ChatGPT Plus gives you no API access at all. If you have a Plus
> subscription, you still have to do the steps below.

## Which provider?

| | **Google Gemini** (course default) | **OpenAI** (optional alternative) |
|---|---|---|
| Cost | Free | ~$5 minimum, prepaid |
| Credit card | Not required | Required |
| Limits | A few requests per minute; occasional waiting | Effectively none at our volume |
| Privacy | Free-tier content may be used to improve Google's models | Not used for training |

**Use Gemini unless you have a reason not to.** It costs nothing, needs no card, and does
everything this course asks. The whole semester fits inside the free tier.

> **Read this before you choose your document.** Google's free tier may use what you send
> it to improve their models. That is the trade for it being free, and it is normal for
> free AI services — but it means the corpus you upload should be non-confidential. Use
> public documents, your own writing, or a substitute. Never upload an employer's
> confidential material, personal data, or anything you would not want seen outside its
> original context. If your real work is interesting but sensitive, build a public
> analogue instead; every skill in this course transfers exactly.

OpenAI is available if you prefer it, or if you hit Gemini's rate limits and find them
annoying. Pick one and stay with it — swapping midway is a two-minute change, but the
screenshots you already took won't match anymore.

---

## Path A — Getting a Gemini key (recommended)

1. Go to **aistudio.google.com** and sign in with a Google account.
2. Click **Get API key** — it is at the top left, or under your profile.
3. Click **Create API key**, then copy it.

That is the whole process. No card, no billing page, no spending cap to set. The key
does not expire.

## Path B — Getting an OpenAI key (optional)

1. Go to **platform.openai.com** and sign in (or create an account — again, separate
   from ChatGPT).
2. **Set your spending cap first.** Profile icon → **Settings** → **Billing** →
   **Limits**. Set a hard monthly budget of $5 and switch **auto-recharge off**.
3. **Billing** → **Add payment details**, and add $5. That is OpenAI's minimum, not an
   estimate of what this course costs — actual usage is a few cents.
4. Left sidebar → **API keys** → **Create new secret key**. Name it `CBIS 5530`.
5. **Copy it immediately.** It is shown once. Lost it? Delete and make another.

---

## Now put the key somewhere safe

**Do not paste your key into a code box.** Anyone you share the notebook with would
then have it, and keys pasted into notebooks are one of the most common ways they leak.
Colab has a built-in vault for exactly this:

1. Click the **🔑 key icon** in the left sidebar of this page.
2. Click **+ Add new secret**.
3. In **Name**, type the name that matches your provider, exactly, capitals included:
   - Gemini → `GEMINI_API_KEY`
   - OpenAI → `OPENAI_API_KEY`
4. In **Value**, paste your key.
5. **Switch on the "Notebook access" toggle.** This is the step everyone forgets, and
   its error message is unhelpful. If Step 2B fails, come back and check this first.

Your key lives in your Google account, not in this file. When you share the notebook,
the key does not travel with it.

---
## Step 2A — Tell the notebook which provider you chose

**This is the box you edit.** It is the only settings box in the notebook, and it is
deliberately short so nothing gets lost in it.

Change the word inside the quote marks on the `PROVIDER` line to match the path you
took above. It already says `"gemini"`, which is what most of you want — change it to
`"openai"` only if you took Path B. Leave the three dials alone for now; you'll
come back and change them in Step 11.

Then press ▶.

In [ ]:
# ============================================================
#  THE ONE LINE YOU CHANGE  -->  "gemini"  (default)  or  "openai"
# ============================================================

PROVIDER = "gemini"

# ============================================================


# The dials you will change in Step 11. Leave them for now.
CHUNK_SIZE    = 1000     # characters per chunk
CHUNK_OVERLAP = 150      # characters repeated between neighbours
TOP_K         = 4        # how many chunks to retrieve per question

print(f"Provider set to: {PROVIDER}")

### Step 2B — Connect

You do not need to change anything in this box. Press ▶.

If it prints **Connection successful**, you are set up and everything below will work.
If it prints an error, read the message — it names the likely cause.

> **Instructor note — verify each term.** The four model names below are the only
> version-sensitive strings in the notebook. Confirm them against the provider's
> current model list before the semester opens. `text-embedding-3-small` has been
> stable and remains the cheapest OpenAI embedding model; the chat model name is the
> one most likely to have moved.

In [ ]:
# --- Model names (instructor-maintained; students need not touch these) ---
OPENAI_CHAT_MODEL  = "gpt-4o-mini"
OPENAI_EMBED_MODEL = "text-embedding-3-small"
GEMINI_CHAT_MODEL  = "gemini-2.5-flash"      # free tier; also try "gemini-3-flash"
GEMINI_EMBED_MODEL = "gemini-embedding-001"  # older alt: "text-embedding-004"

import numpy as np
from google.colab import userdata

if PROVIDER == "openai":
    SECRET_NAME = "OPENAI_API_KEY"
elif PROVIDER == "gemini":
    SECRET_NAME = "GEMINI_API_KEY"
else:
    raise ValueError(
        f'PROVIDER is "{PROVIDER}". It must be exactly "openai" or "gemini" '
        f"(lower case, in quote marks). Fix it in Step 2A and run this again."
    )

try:
    api_key = userdata.get(SECRET_NAME)
except Exception:
    api_key = None

if not api_key:
    raise SystemExit(
        f"No key found under the name {SECRET_NAME}.\n\n"
        f"Check three things, in this order:\n"
        f"  1. Click the key icon in the left sidebar. Is there a secret named\n"
        f"     exactly {SECRET_NAME}? Capitals and underscores matter.\n"
        f"  2. Is its Notebook access toggle switched ON? This is the usual cause.\n"
        f"  3. Does the name match the PROVIDER you set in Step 2A?"
    )

if PROVIDER == "openai":
    from openai import OpenAI
    client = OpenAI(api_key=api_key)
    CHAT_MODEL, EMBED_MODEL = OPENAI_CHAT_MODEL, OPENAI_EMBED_MODEL
else:
    from google import genai
    client = genai.Client(api_key=api_key)
    CHAT_MODEL, EMBED_MODEL = GEMINI_CHAT_MODEL, GEMINI_EMBED_MODEL

import time


def with_retry(fn, *a, **kw):
    # Free tiers are rate-limited. Wait and retry rather than crashing.
    delays = [2, 5, 10, 20, 40]
    for n, wait in enumerate(delays + [None]):
        try:
            return fn(*a, **kw)
        except Exception as e:
            msg = str(e).lower()
            if "not found" in msg or "404" in msg:
                raise SystemExit(
                    "The model name was not recognised.\n\n"
                    "Model names change every few months. Check your provider\n"
                    "for the current name, then update Step 2B.\n"
                    "  Gemini: aistudio.google.com    OpenAI: platform.openai.com"
                )
            limited = ("429" in msg or "rate" in msg or "quota" in msg
                       or "resource_exhausted" in msg)
            if not limited or wait is None:
                raise
            print(f"  Rate limit reached. Waiting {wait}s, then retrying "
                  f"({n+1} of {len(delays)})...")
            time.sleep(wait)


print(f"Provider:        {PROVIDER}")
print(f"Chat model:      {CHAT_MODEL}")
print(f"Embedding model: {EMBED_MODEL}")
print(f"Key found under: {SECRET_NAME}")
print()
print("Connection successful. Continue to Step 3.")

---
## Step 3 — Load a document

Run the box below and a **Choose Files** button appears. Pick one file from your
computer. The notebook accepts **PDF (.pdf), Word (.docx), and plain text (.txt,
.md)**.

Start with something short — 5 to 30 pages. A policy document, a report, a chapter.
Something you already know well, so you can tell when the answers are wrong.

**Two file types that will not work, and what to do:**

- **Old Word files (.doc)** — open in Word and do *File → Save As → .docx* first.
- **Scanned PDFs** — these are pictures of text, not text. The notebook will tell you
  if you've hit one. Use a different document; making scans readable is a separate
  problem called OCR and is out of scope here.

> The book's phrase for the whole category is *garbage in, garbage out* (§3.7,
> discussion 1). Document collection sets the ceiling on everything downstream — if
> the text never loads cleanly, no amount of clever retrieval can rescue it.

In [ ]:
# A5 CHANGE: accept several files (both books) and join them in file-name order.
# The per-format logic is the course loader, unchanged.
from google.colab import files
import io, os

def read_one(filename, raw):
    ext = os.path.splitext(filename)[1].lower()
    if ext == ".pdf":
        from pypdf import PdfReader
        reader = PdfReader(io.BytesIO(raw))
        return "\n".join((page.extract_text() or "") for page in reader.pages)
    if ext == ".docx":
        import docx
        d = docx.Document(io.BytesIO(raw))
        parts = [p.text for p in d.paragraphs]
        for table in d.tables:
            for row in table.rows:
                parts.append(" | ".join(cell.text for cell in row.cells))
        return "\n".join(parts)
    if ext in (".txt", ".md"):
        return raw.decode("utf-8", errors="ignore")
    raise SystemExit(f"{filename}: unsupported file type {ext}")

uploaded = files.upload()
FILE_ORDER = sorted(uploaded)                 # Book1_... sorts before Book2_...
texts = [read_one(f, uploaded[f]) for f in FILE_ORDER]
document_text = "\n\n".join(texts)

# Remember where each file starts, so retrieved chunks can be labeled by book.
FILE_STARTS, pos = [], 0
for f, t in zip(FILE_ORDER, texts):
    FILE_STARTS.append((pos, f))
    pos += len(t) + 2

for f, t in zip(FILE_ORDER, texts):
    print(f"Read {len(t):,} characters from {f}")
print(f"Combined: {len(document_text):,} characters")


---
# ARTIFACT 1 — Chunks
## Step 4 — Cut the document into pieces

The model cannot read your whole document for every question, and you would not want
it to. So the document is cut into passages. The book calls this **chunking** (§3.2).

The method below is the simplest one in the book: **fixed-size chunking**. Count off
`CHUNK_SIZE` characters, cut, repeat. It pays no attention to sentences, headings, or
meaning — which is exactly its weakness, and exactly the point of a later assignment.

`CHUNK_OVERLAP` makes each piece repeat a little of the one before it, so a sentence
sitting on a cut line still appears whole somewhere. The book calls this **chunk
overlap**, and the HR story at the start of Chapter 3 is what happens without it.

In [ ]:
# --- Did you run Step 2A? ---
if "CHUNK_SIZE" not in globals() or "CHUNK_OVERLAP" not in globals():
    raise SystemExit(
        "The chunk settings are not defined yet, which means Step 2A has not run\n"
        "in this session. Scroll up and run it, then come back.\n\n"
        "Fastest fix: Runtime > Run all. Then return here.\n\n"
        "IMPORTANT: do not paste the settings into this box to make the error go\n"
        "away. They must live in Step 2A only, or Step 11 will not work."
    )


def chunk_text(text, size, overlap):
    """Cut text into overlapping fixed-size pieces. This is the whole algorithm."""
    if overlap >= size:
        raise ValueError("CHUNK_OVERLAP must be smaller than CHUNK_SIZE")
    pieces, start = [], 0
    while start < len(text):
        pieces.append(text[start:start + size])
        start += size - overlap
    return [p.strip() for p in pieces if p.strip()]


chunks = chunk_text(document_text, CHUNK_SIZE, CHUNK_OVERLAP)

print(f"Your document became {len(chunks)} chunks.")
print(f"Settings used: CHUNK_SIZE={CHUNK_SIZE}, CHUNK_OVERLAP={CHUNK_OVERLAP}")
print()
print("=" * 60)
print("CHUNK 0 (the first one)")
print("=" * 60)
print(chunks[0])
print()
print("=" * 60)
print("CHUNK 1 (the second one)")
print("=" * 60)
print(chunks[1] if len(chunks) > 1 else "(your document only made one chunk)")
print()
print("Look at where chunk 0 ends and chunk 1 begins.")
print("Did it cut a sentence in half? That is fixed-size chunking doing its job badly.")

---
# ARTIFACT 2 — An embedding
## Step 5 — Turn one chunk into numbers

This is the step most people find strangest, so we do it once, slowly, on a single
chunk before doing it to all of them.

An **embedding** (§2.1) is a list of numbers that stands for the *meaning* of a piece
of text. Texts that mean similar things get similar lists. That is the entire trick
that lets a search for "I can't log in" find an article titled "Resolving
authentication failures."

Run the box and look at what comes back. It is not a summary. It is not keywords. It
is a few hundred or a few thousand numbers, and none of them means anything on its
own. Only the *position* they describe means something.

In [ ]:
# A5 CHANGE: embed in batches. One call with ~650 chunks exceeds the provider's
# per-request limit. Same model, same vectors, just sent in groups.
EMBED_BATCH = 50

def embed_texts(texts):
    """Turn a list of strings into a list of embeddings."""
    out = []
    for i in range(0, len(texts), EMBED_BATCH):
        part = texts[i:i + EMBED_BATCH]
        if PROVIDER == "openai":
            resp = with_retry(client.embeddings.create, model=EMBED_MODEL, input=part)
            out.extend(d.embedding for d in resp.data)
        else:
            resp = with_retry(client.models.embed_content, model=EMBED_MODEL,
                              contents=part)
            out.extend(e.values for e in resp.embeddings)
    return out


sample = chunks[0]
sample_vector = embed_texts([sample])[0]
print(f"A list of {len(sample_vector)} numbers. First ten:")
print([round(n, 4) for n in sample_vector[:10]])


---
## Step 6 — Embed everything (this is your vector store)

Now the same thing to every chunk. This costs a fraction of a cent and takes a few
seconds. This is the **offline phase** the book describes in §3.1 — the work done
once, in advance, before any question is asked.

What you get is a grid: one row per chunk, one column per number in the embedding.

**This grid is your vector database.** Chapter 2 describes specialised databases with
HNSW and IVF indexes, and at scale you need them. At *this* scale you do not — we
compare against every row every time. The book lists that honestly in Chapter 3,
Table 1, as **Flat (exact) search**: guarantees the true nearest neighbours, too slow
at scale, perfect for small collections. You are not skipping a step. You are using
the one the book recommends for a corpus this size.

In [ ]:
# --- Safety check: this is the only step that costs money. ---
MAX_CHUNKS = 1000   # A5 CHANGE: two books at 1000/150 make ~660 chunks
est_tokens = sum(len(c) for c in chunks) / 4        # ~4 characters per token
est_cost   = est_tokens / 1_000_000 * 0.02          # text-embedding-3-small rate

print(f"About to embed {len(chunks)} chunks (~{est_tokens:,.0f} tokens).")
print(f"Estimated cost: ${est_cost:.4f}")

if len(chunks) > MAX_CHUNKS:
    raise SystemExit(
        f"STOPPED. {len(chunks)} chunks is more than expected for this course.\n"
        f"Usually this means CHUNK_OVERLAP is too close to CHUNK_SIZE, or the\n"
        f"document is very long. Go back to Step 2, lower CHUNK_OVERLAP or raise\n"
        f"CHUNK_SIZE, re-run Step 4, and try again. Nothing has been charged."
    )

vectors = np.array(embed_texts(chunks))

print()
print(f"Shape of your vector store: {vectors.shape}")
print(f"  -> {vectors.shape[0]} chunks")
print(f"  -> {vectors.shape[1]} numbers describing each one")
print()
print("That is the whole knowledge base. It is a grid of numbers.")

---
# ARTIFACT 3 — Retrieved passages
## Step 7 — Ask a question and watch what comes back

Now the **online phase** (§3.1) — the part that runs every time someone asks
something. Your question gets embedded by the *same model* that embedded the chunks,
so both live in the same space and can be compared. (§3.2 calls that the one rule
that must not be broken.)

Then we measure which chunks sit closest, using **cosine similarity** (§2.3).

**Change the question below to something about your own document.** Then run it and
read the passages that come back *before* you read any answer. This is the moment
where you find out whether your system can work at all — the model cannot use what
was never retrieved.

In [ ]:
question = "What is this document about?"   # <-- CHANGE THIS


def retrieve(question, k=TOP_K):
    """Return the k chunks whose meaning sits closest to the question."""
    q = np.array(embed_texts([question])[0])
    # cosine similarity = dot product of the direction each vector points
    sims = (vectors @ q) / (np.linalg.norm(vectors, axis=1) * np.linalg.norm(q))
    top = np.argsort(sims)[::-1][:k]
    return [(int(i), float(sims[i]), chunks[i]) for i in top]


hits = retrieve(question)

print(f"QUESTION: {question}")
print(f"Searched {len(chunks)} chunks, keeping the top {TOP_K}.")
print()
for rank, (idx, score, text) in enumerate(hits, start=1):
    print("=" * 60)
    print(f"RANK {rank}  |  chunk #{idx}  |  similarity {score:.3f}")
    print("=" * 60)
    print(text[:400], "...")
    print()

print("Similarity runs from -1 to 1. Higher is closer in meaning.")
print("Are these actually the right passages? Similar is not the same as relevant (Ch4).")

---
# ARTIFACT 4 — The assembled prompt
## Step 8 — See exactly what the model will be handed

This is the box worth staring at. Everything up to now was preparation; this is the
actual input the language model receives. The book calls the step **context
augmentation** (§3.5).

Notice the two instructions in it. §3.5 says these two do the heaviest lifting in the
whole system:

- *answer only from the context provided*
- *if the context does not contain the answer, say so*

Together they turn the model from a confident guesser into a grounded reader. The HR
assistant that answered "ten" instead of "five" probably had a prompt without them.

Run this and **read the printout end to end.** Most of what confuses people about
these systems stops being confusing once they have read one assembled prompt.

In [ ]:
def build_prompt(question, hits):
    context = "\n\n---\n\n".join(
        f"[Source: chunk #{idx}]\n{text}" for idx, _, text in hits
    )
    return f"""You are an assistant answering from supplied context only.

Using the context below, answer the question at the end. If the context does not
contain the answer, say you do not have enough information. Cite the chunk
numbers you used.

CONTEXT:
{context}

QUESTION: {question}
"""


prompt = build_prompt(question, hits)

print("=" * 60)
print("THIS IS EXACTLY WHAT THE MODEL RECEIVES")
print("=" * 60)
print(prompt)
print("=" * 60)
print(f"Length: {len(prompt):,} characters")

---
## Step 9 — Get the grounded answer

Finally, send that prompt to the model. Nothing clever happens here — the cleverness
already happened, in choosing what to put in front of it.

In [ ]:
def generate(prompt):
    if PROVIDER == "openai":
        r = with_retry(client.chat.completions.create,
                       model=CHAT_MODEL,
                       messages=[{"role": "user", "content": prompt}])
        return r.choices[0].message.content
    else:
        r = with_retry(client.models.generate_content, model=CHAT_MODEL,
                       contents=prompt)
        return r.text


print(generate(prompt))

---
# A5 — Run the test set

Runs all ten questions once with the settings in Step 2A, and saves three files:

- `a5_results.json` — everything, for the record
- `a5_retrieval_dump.md` — each question, its four retrieved chunks in full (with book and section labels), and the answer. **Score from this file.**
- `a5_results.csv` — one row per question for the spreadsheet

The book/section label is added to the dump only, to make hand scoring faster. The model never sees it.


---
# FINAL PROJECT: classic RAG re-run with extra measurements

Same settings and same ten questions as A5. Adds: thinking tokens, embedding-call time vs numpy search time, retrieved characters, and an embedding-token estimate. Also a second run of the same system, which shows how much the answers move between runs. Saves `classic_rerun_*` files (A5 files are untouched).

In [ ]:
import json, csv, re, time

QUESTIONS = [
    ("Q1", "According to the research cited in AI's Missing Playbook, what share of organizations use AI in at least one function, and what share of enterprise AI projects produced no measurable effect on the bottom line?"),
    ("Q2", "What are the seven parts of a strong prompt?"),
    ("Q3", "In the three-step customer operations example, what is the Rolled Throughput Yield and the cost per verified chain completion?"),
    ("Q4", "How many evaluation trials are needed to confirm a 90% First Pass Yield at 95% confidence with a margin of error of plus or minus 5 points, and how many at plus or minus 2 points?"),
    ("Q5", "In the document review ledger, what did the organization budget per month versus actually spend, and what share of the true per-file cost was human review?"),
    ("Q6", "What does \"the machinist, not the machine\" mean as an AI architecture?"),
    ("Q7", "In AI's Missing Playbook, what is the third option besides build and buy, and what rule governs it?"),
    ("Q8", "What must the ninety-day proof point deliver, and which level of the AI Maturity Self-Assessment does achieving it correspond to?"),
    ("Q9", "What chunk size, in characters or tokens, does AI's Missing Playbook recommend for a RAG pipeline?"),
    ("Q10", "What are the three tiers of the Three-Tier Enterprise Data Taxonomy?"),
]

# Label each chunk with its book and the nearest H1 / H2 above its start.
HEADINGS = [(m.start(), len(m.group(1)), m.group(2).strip())
            for m in re.finditer(r"^(#{1,2}) (.+)$", document_text, re.M)]

def label(idx):
    start = idx * (CHUNK_SIZE - CHUNK_OVERLAP)
    book = [f for s, f in FILE_STARTS if s <= start][-1].split("_")[0]
    h1 = h2 = ""
    for off, lvl, text in HEADINGS:
        if off > start:
            break
        if lvl == 1:
            h1, h2 = text, ""
        else:
            h2 = text
    return f"{book} > {h1}" + (f" > {h2}" if h2 else "")

def generate_with_usage(prompt):
    if PROVIDER == "openai":
        r = with_retry(client.chat.completions.create, model=CHAT_MODEL,
                       messages=[{"role": "user", "content": prompt}])
        return r.choices[0].message.content, r.usage.prompt_tokens, r.usage.completion_tokens
    r = with_retry(client.models.generate_content, model=CHAT_MODEL, contents=prompt)
    u = r.usage_metadata
    global LAST_THOUGHTS
    LAST_THOUGHTS = getattr(u, "thoughts_token_count", 0) or 0
    return r.text, u.prompt_token_count, u.candidates_token_count

results = []
for qid, q in QUESTIONS:
    te = time.time(); _qv = embed_texts([q])[0]; t_embed = time.time() - te
    t0 = time.time(); hits = retrieve(q); t1 = time.time()
    prompt = build_prompt(q, hits)
    answer, tin, tout = generate_with_usage(prompt); t2 = time.time()
    results.append({
        "id": qid, "question": q, "answer": answer,
        "retrieval_s": round(t1 - t0, 2), "embed_call_s": round(t_embed, 2), "thought_tokens": LAST_THOUGHTS,
        "context_chars": sum(len(t) for _, _, t in hits), "generation_s": round(t2 - t1, 2),
        "prompt_tokens": tin, "answer_tokens": tout,
        "hits": [{"rank": r + 1, "chunk": i, "similarity": round(s, 3),
                  "location": label(i), "text": t} for r, (i, s, t) in enumerate(hits)],
    })
    print(f"{qid} done  (retrieval {t1-t0:.1f}s, generation {t2-t1:.1f}s)")

settings = dict(provider=PROVIDER, chat_model=CHAT_MODEL, embed_model=EMBED_MODEL,
                chunk_size=CHUNK_SIZE, chunk_overlap=CHUNK_OVERLAP, top_k=TOP_K,
                n_chunks=len(chunks), files=FILE_ORDER, embed_tokens_est=sum(len(c) for c in chunks)//4, run_at=time.strftime("%Y-%m-%d %H:%M"))
json.dump({"settings": settings, "results": results}, open("classic_rerun_results.json", "w"), indent=2)

with open("classic_rerun_dump.md", "w") as f:
    f.write(f"# A5 retrieval dump\n\nSettings: {settings}\n\n")
    for r in results:
        f.write(f"\n---\n\n## {r['id']}: {r['question']}\n\n")
        for h in r["hits"]:
            f.write(f"### Rank {h['rank']} | chunk #{h['chunk']} | sim {h['similarity']} | {h['location']}\n\n")
            f.write("```text\n" + h["text"] + "\n```\n\n")
        f.write(f"### Answer\n\n{r['answer']}\n\n")
        f.write(f"*Retrieval {r['retrieval_s']}s, generation {r['generation_s']}s, "
                f"{r['prompt_tokens']} prompt tokens, {r['answer_tokens']} answer tokens*\n")

with open("classic_rerun_results.csv", "w", newline="") as f:
    w = csv.writer(f)
    w.writerow(["id", "question", "rank1_chunk", "rank2_chunk", "rank3_chunk", "rank4_chunk",
                "retrieval_s", "generation_s", "prompt_tokens", "answer_tokens", "answer"])
    for r in results:
        w.writerow([r["id"], r["question"], *[h["chunk"] for h in r["hits"]],
                    r["retrieval_s"], r["generation_s"], r["prompt_tokens"],
                    r["answer_tokens"], r["answer"]])

for name in ("classic_rerun_results.json", "classic_rerun_dump.md", "classic_rerun_results.csv"):
    files.download(name)
print("Saved and downloaded. Saved classic_rerun_* files.")


---
## What you just built

You now have, running in a browser tab, a complete retrieval-augmented generation
pipeline. Both phases from §3.1 are in front of you:

**Offline** — load, chunk, embed, store. Steps 3 through 6.
**Online** — embed the question, retrieve, augment, generate. Steps 7 through 9.

It has no reranker (Ch4), no hybrid search (§3.4), no evaluation (Ch8), and no
access control (Ch9). The book will name all of those. Now you have somewhere to
put them.

*Georgia College & State University · J. Whitney Bunting College of Business &
Technology · CBIS 5530*